# Zonas de vida de Holdridge (38 zonas) — CHELSA V2.1, normal 1991-2020

Versão em Python (API do Earth Engine) do script JS anterior, adaptada aos **três assets separados por
variável** (`tas`, `pet`, `pr`; 12 bandas mensais cada), gerados por `gerar_normal_multibanda.py`.

**O que mudou em relação ao script antigo:** os assets já estão em unidades físicas
(tas em °C; pet e pr em mm/mês). Por isso **não há mais** `subtract(273.15)`, `divide(100)` nem `divide(10)`.
A biotemperatura, a classificação por classes e a tabela das 38 zonas seguem a lógica do script original.
O cálculo está em [holdridge_gee.py](holdridge_gee.py).

A suavização por moda (focalMode) do script JS foi trazida de volta (seção 3b): `brasil` (contorno real do
país, FAO GAUL) é usado para `clip`/geometria/mapa e também para recortar `zona_final` de volta ao contorno
do país depois da suavização (que pode espalhar valores um pouco além da borda). `zona_final` tem as mesmas
características do que seria exportado manualmente no GEE. Este notebook **não exporta** nada
automaticamente; a exportação fica a cargo do usuário.

## 1. Configuração

In [1]:
import sys
sys.path.insert(0, ".")

import ee
import geemap
import pandas as pd
import holdridge_gee as h

PROJETO = "fcoliveira"

# Assets separados por variavel (12 bandas mensais cada; ajuste para o caminho onde voce subiu as imagens)
ASSET_TAS = "projects/fcoliveira/assets/chelsa_brasil_tas_normal_1991_2020"
ASSET_PET = "projects/fcoliveira/assets/chelsa_brasil_pet_normal_1991_2020"
ASSET_PR = "projects/fcoliveira/assets/chelsa_brasil_pr_normal_1991_2020"

# Asset de saida com a classificacao (referencia; a exportacao e manual, ver secao 6)
ASSET_SAIDA = "projects/fcoliveira/assets/CHELSA/Holdridge_CHELSA_BR_1991_2020"

# None = correcao de latitude em todos os meses (igual ao script JS original).
# 24  = correcao so nos meses com t > 24 C. Veja a secao 4 antes de decidir.
LIMIAR_CORRECAO = None


## 2. Earth Engine e região (Brasil)

In [2]:
try:
    ee.Initialize(project=PROJETO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO)

# 'brasil' usa o contorno real do pais (FAO GAUL) para clip/geometria/mapa.
brasil = (ee.FeatureCollection("FAO/GAUL/2015/level0")
          .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))
estados = (ee.FeatureCollection("FAO/GAUL/2015/level1")
           .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## 3. Carregar a normal e classificar

In [3]:
normal = h.carregar_normal(ASSET_TAS, ASSET_PET, ASSET_PR)
print("Bandas:", normal.bandNames().getInfo())

resultado = h.classificar_holdridge(normal, LIMIAR_CORRECAO).clip(brasil)
zona = resultado.select("zone38_id")

Bandas: ['tas_01', 'tas_02', 'tas_03', 'tas_04', 'tas_05', 'tas_06', 'tas_07', 'tas_08', 'tas_09', 'tas_10', 'tas_11', 'tas_12', 'pet_01', 'pet_02', 'pet_03', 'pet_04', 'pet_05', 'pet_06', 'pet_07', 'pet_08', 'pet_09', 'pet_10', 'pet_11', 'pet_12', 'pr_01', 'pr_02', 'pr_03', 'pr_04', 'pr_05', 'pr_06', 'pr_07', 'pr_08', 'pr_09', 'pr_10', 'pr_11', 'pr_12']


## 3b. Suavização (focalMode) e recorte final

Reproduz as etapas finais do script JS original: filtro de moda 3x3 na zona classificada e recorte de
volta ao contorno de `brasil` (a suavização pode espalhar valores por 1 pixel além da borda; o `clip`
final remove essa sobra e também define como "sem zona" (0) os pixels dentro do país que não receberam
classificação). O resultado, `zona_final`, é a imagem pronta para a exportação manual (seção 6).

In [4]:
zona_final = (h.suavizar_zona(zona).unmask(0).clip(brasil)
              .rename("zone38_id")
              .toByte())


## 4. Verificação: áreas por zona

No Brasil não se espera gelo/polar (zonas 1 e 2). **Se aparecerem em grande área no Sul, é a correção de latitude
aplicada em todos os meses**: com temperaturas em °C corretas, `t - 0,03 * lat * (t - 24)^2` derruba a temperatura
de meses mais frios a zero em latitudes maiores que ~25°. Nesse caso, defina `LIMIAR_CORRECAO = 24` na configuração
e rode de novo.

In [5]:
hist = zona_final.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=brasil.geometry(),
    scale=5000,
    maxPixels=1e13,
    bestEffort=True,
).get("zone38_id").getInfo()

tabela = (pd.Series(hist, name="pixels_5km").rename_axis("zona").reset_index()
          .assign(zona=lambda d: d.zona.astype(int))
          .sort_values("zona"))
tabela["area_km2"] = tabela["pixels_5km"] * 25
tabela["pct"] = (100 * tabela["pixels_5km"] / tabela["pixels_5km"].sum()).round(2)
display(tabela)

if tabela["zona"].isin([1, 2]).any():
    pct = tabela.loc[tabela["zona"].isin([1, 2]), "pct"].sum()
    print(f"ATENCAO: {pct:.1f}% do Brasil caiu nas zonas 1-2 (gelo/polar). Reveja LIMIAR_CORRECAO.")


,zona,pixels_5km,area_km2,pct
0,0,8.850980,2.212745e+02,0.00
1,1,1444.000000,3.610000e+04,0.41
7,2,154.000000,3.850000e+03,0.04
24,4,2.000000,5.000000e+01,0.00
25,5,1021.000000,2.552500e+04,0.29
26,6,159.000000,3.975000e+03,0.05
2,10,4479.086275,1.119772e+05,1.28
3,11,84.356863,2.108922e+03,0.02
4,15,2865.423529,7.163559e+04,0.82
5,16,16276.458824,4.069115e+05,4.64


ATENCAO: 0.4% do Brasil caiu nas zonas 1-2 (gelo/polar). Reveja LIMIAR_CORRECAO.


## 5. Mapa

In [6]:
vis = {"min": 1, "max": 38, "palette": h.PALETA}

Map = geemap.Map(center=[-14, -52], zoom=4)
Map.addLayer(zona, vis, "Holdridge (38 zonas)")
Map.addLayer(zona_final, vis, "Holdridge (38 zonas kernel)")
Map.addLayer(resultado.select("biotemp"), {"min": 10, "max": 30, "palette": ["blue", "yellow", "red"]}, "Biotemperatura", False)
Map.addLayer(resultado.select("prec"), {"min": 0, "max": 3500, "palette": ["white", "blue"]}, "Precipitacao anual", False)
Map.addLayer(resultado.select("retp"), {"min": 0, "max": 4, "palette": ["green", "yellow", "red"]}, "Razao ETP/P", False)
Map.addLayer(estados.style(color="000000", fillColor="00000000", width=1), {}, "Estados")
Map.addLayer(brasil.style(color="000000", fillColor="00000000", width=2), {}, "Brasil")
Map.add_colorbar(vis, label="Zona de Holdridge (id 1-38)", layer_name="Holdridge (38 zonas)")
Map


Map(center=[-14, -52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

## 6. Exportação

A célula abaixo baixa `zona_final` como GeoTIFF local, na resolução nativa do CHELSA (~928 m, a mesma
dos assets de entrada) e recortado por `brasil`. O arquivo sai em `climas/dados_chelsa/holdridge/`, no
mesmo padrão do que `gerar_normal_multibanda.py` já faz para tas/pet/pr: gera local e depois sobe manualmente
para o GEE (via bucket do GCS, igual ao fluxo em `chelsa_brasil.ipynb`).

In [ ]:
from pathlib import Path

SAIDA_DIR = Path("../dados_chelsa/holdridge")
SAIDA_DIR.mkdir(parents=True, exist_ok=True)
caminho_local = SAIDA_DIR / "Holdridge_CHELSA_BR_1991_2020.tif"

proj = normal.select(0).projection()

# download_ee_image (nao ee_export_image) porque a imagem passa do limite de 48MB de
# download direto do GEE; ele baixa em tiles e remonta um unico GeoTIFF.
geemap.download_ee_image(
    zona_final,
    filename=str(caminho_local),
    region=brasil.geometry(),
    crs=proj.crs().getInfo(),
    scale=proj.nominalScale().getInfo(),
    dtype="uint8",
)

print(f"Salvo em: {caminho_local.resolve()}")
print("\nDepois, subir para um bucket no GCS e criar o asset (mesmo fluxo do chelsa_brasil.ipynb):")
print(f"  gsutil cp {caminho_local} gs://SEU_BUCKET/Holdridge_CHELSA_BR_1991_2020.tif")
print(f"  earthengine upload image --asset_id={ASSET_SAIDA} --pyramiding_policy=mode "
      "gs://SEU_BUCKET/Holdridge_CHELSA_BR_1991_2020.tif")
